In [5]:
!pip install -r requirements-train.txt

  Cloning https://github.com/Keeyahto/CLIP-ONNX.git to /tmp/pip-req-build-_mblnijk
  Running command git clone --filter=blob:none --quiet https://github.com/Keeyahto/CLIP-ONNX.git /tmp/pip-req-build-_mblnijk
  Resolved https://github.com/Keeyahto/CLIP-ONNX.git to commit c13158db1d947a432cb36ffa6f7b83850651ba90
  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'clip_onnx' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'clip_onnx'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for clip_onnx: filename=clip_onnx-1.2-py3-none-any.whl size=5783 sha256=85aae8214239c0dfe80665cfb6f1cd8a4a8d60ee993cf2ea0f65f7026e71c9ac
  Stored in directory: /tmp/pip-

In [32]:
!python -m open_clip_train.main --save-frequency 25 --batch-size 8 --workers 1 --train-data 'samples-00.tar' --train-num-samples 10  --dataset-type webdataset 

2026-03-20,19:36:43 | INFO | Running with a single process. Device cuda.
2026-03-20,19:36:43 | INFO | Parsing model identifier. Schema: None, Identifier: RN50
2026-03-20,19:36:43 | INFO | Loaded built-in RN50 model config.
2026-03-20,19:36:43 | INFO | No potential checkpoint path found from config source or pretrained arg.
2026-03-20,19:36:43 | INFO | Instantiating model architecture: CLIP
/opt/app-root/lib64/python3.12/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
2026-03-20,19:36:44 | WARNING | No pretrained weights loaded for model 'RN50'. Model initialized randomly.
2026-03-20,19:36:44 | INFO | Final image preprocessing configuration set: {'size': 224, 'mode': 'RGB', 'mean': (0.48145466, 0.4578275, 0.40821073), 'std': (0.26862954, 0.26130258, 0.27577711), 'interpolation': 'bicubic', 'resize_mode': 'shortest', 'fill_color': 0}
2026-03-20,19:36:44 | INFO | Model RN50 creation process complete.
2026-03-20,19:36:44

In [ ]:
import torch
import open_clip
from PIL import Image
from clip_onnx import clip_onnx

visual_path = "clip_visual.onnx"
textual_path = "clip_textual.onnx"

model, _, preprocess = open_clip.create_model_and_transforms('RN50', pretrained='./logs/2026_03_20-19_30_09-model_RN50-lr_0.0005-b_8-j_1-p_amp/checkpoints/epoch_32.pt') 

# dummy tensors
image = preprocess(Image.open("CLIP.png")).unsqueeze(0).cpu() # [1, 3, 224, 224]
text = open_clip.tokenize(["a diagram"]).cpu()

onnx_model = clip_onnx(model, visual_path=visual_path, textual_path=textual_path)
onnx_model.convert2onnx(image, text, verbose=True)


[CLIP ONNX] Start convert visual model
